<a href="https://colab.research.google.com/github/avgvstinee/python-ml-essentials/blob/main/Untitled2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Define Residual Block

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from typing import Any, Callable, List, Optional

def conv3x3(in_planes: int, out_planes: int, stride: int = 1,
            padding: int = 1) -> nn.Conv2d:
    """3x3 convolution with padding"""
    return nn.Conv2d(
        in_planes,
        out_planes,
        kernel_size=3,
        stride=stride,
        padding=padding,
        bias=False
    )

class BasicBlock(nn.Module):
    expansion: int = 1

    def __init__(
        self,
        inplanes: int,
        planes: int,
        stride: int = 1,
        downsample: Optional[nn.Module] = None,
        norm_layer: Optional[Callable[..., nn.Module]] = None,
    ) -> None:
        super().__init__()
        if norm_layer is None:
            norm_layer = nn.BatchNorm2d

        self.conv1 = conv3x3(inplanes, planes, stride)
        self.bn1 = norm_layer(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(planes, planes)
        self.bn2 = norm_layer(planes)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x: Tensor) -> Tensor:
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out

# Define ResNet18 Architecture

In [ ]:
class ResNet18(nn.Module):
    def __init__(self, num_classes=100, dropout_p=0.2):
        super().__init__()
        # Stem — small 3×3 conv for CIFAR 32×32 (no maxpool)
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.layer1 = self._make_layer(64,  64,  2, stride=1)
        self.layer2 = self._make_layer(64,  128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.layer4 = self._make_layer(256, 512, 2, stride=2)
        self.pool    = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(p=dropout_p)
        self.fc      = nn.Linear(512, num_classes)
        self._init_weights()

    def _make_layer(self, in_ch, out_ch, n_blocks, stride):
        downsample = None
        if stride != 1 or in_ch != out_ch:
            downsample = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )

        layers = [BasicBlock(in_ch, out_ch, stride=stride, downsample=downsample)]
        for _ in range(1, n_blocks):
            layers.append(BasicBlock(out_ch, out_ch, stride=1))
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.stem(x)                  # (B,64,32,32)
        x = self.layer1(x)                # (B,64,32,32)
        x = self.layer2(x)                # (B,128,16,16)
        x = self.layer3(x)                # (B,256,8,8)
        x = self.layer4(x)                # (B,512,4,4)
        x = self.pool(x).flatten(1)       # (B,512)
        x = self.dropout(x)
        return self.fc(x)                 # (B,100)


# Install and Import Libraries

In [9]:
!pip install torch torchvision matplotlib -q
!pip3 uninstall --yes torch torchaudio torchvision torchtext torchdata
!pip3 install torch torchaudio torchvision torchtext torchdata

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchdata 0.11.0
Uninstalling torchdata-0.11.0:
  Successfully uninstalled torchdata-0.11.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.5 MB/s eta 0:00:00


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torch.utils.data import random_split
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

# 1. Device Setup for Mac
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("Using MPS (Mac GPU)")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print("Using CUDA (NVIDIA GPU)")
else:
    DEVICE = torch.device("cpu")
    print("Using CPU (Warning: This will be slow)")

Using CUDA (NVIDIA GPU)


# Load and Preprocess CIFAR-100 Dataset

In [2]:

# Data Augmentation

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408),(0.2675, 0.2565, 0.2761))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408),(0.2675, 0.2565, 0.2761))
    ])


full_train_set = torchvision.datasets.CIFAR100(root='./data',
                                                   train=True,
                                                   download=True,
                                                   transform=transform_train
                                                   )
test_set = torchvision.datasets.CIFAR100(root='./data',
                                             train=False,
                                             download=True,
                                             transform=transform_test
                                             )

# Split 10% for validation
val_size = 5000
train_size = len(full_train_set) - val_size
train_set, val_set = random_split(full_train_set,
                                          [train_size, val_size]
                                          )



train_loader = DataLoader(train_set, batch_size=128,
                          shuffle=True,
                          num_workers=4,
                          pin_memory=True)
val_loader   = DataLoader(val_set,
                          batch_size=128,
                          shuffle=False,
                          num_workers=4,
                          pin_memory=True)


100%|██████████| 169M/169M [00:02<00:00, 60.6MB/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


# Train the Model

In [3]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
    return running_loss/len(loader), 100.*correct/total

# Validate the Model

In [4]:
def validate(model, loader, criterion):
    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    return val_loss/len(loader), 100.*correct/total

# Main Hyperparamter Search

In [5]:
best_val_acc = 0
best_history = None

# Early Stoping

In [6]:
import copy
class EarlyStopping:
    def __init__(self, patience=10, min_delta=1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self.counter    = 0
        self.best_loss  = float("inf")
        self.best_epoch = 0
        self.best_wts   = None
        self.stop       = False

    def step(self, val_loss, epoch, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss  = val_loss
            self.best_epoch = epoch
            self.best_wts   = copy.deepcopy(model.state_dict())
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                model.load_state_dict(self.best_wts)   # restore best weights
                self.stop = True

In [ ]:

for lr in [0.001, 0.01]:
    for wd in [1e-4, 5e-4]:
        print(f"Starting to train: LR={lr}, WD={wd}")
        # Note: Ensure MyResNet() has softmax REMOVED from forward()

        model = ResNet18().to(DEVICE)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.SGD(model.parameters(),
                               lr=lr,
                               momentum=0.9,
                               weight_decay=wd,
                               nesterov=True
                               )

        scheduler = optim.lr_scheduler.StepLR(optimizer,
                                              step_size=30,
                                              gamma=0.1)


        history = {'training_loss': [], 'training_acc': [],
                   'validation_loss': [], 'validation_acc': []}

        early_stopping = EarlyStopping(patience=7, min_delta=1e-4)

        for epoch in range(100):
            training_loss, training_acc = train_one_epoch(model,
                                            train_loader,
                                            criterion,
                                            optimizer)
            validation_loss, validation_acc = validate(model, val_loader, criterion)
            scheduler.step()

            history['training_loss'].append(training_loss); history['training_acc'].append(training_acc)
            history['validation_loss'].append(validation_loss); history['validation_acc'].append(validation_acc)

            early_stopping.step(validation_loss, epoch, model)

            if validation_acc > best_val_acc:
                best_val_acc = validation_acc
                best_history = history.copy()
                torch.save(model.state_dict(), 'best_model.pth')

            if (epoch + 1) % 5 == 0:
                print(f"Epoch {epoch+1}: Train Loss {training_loss:.4f}, Train Acc {training_acc:.2f}%, Val Loss {validation_loss:.4f}, Val Acc {validation_acc:.2f}%, Dropout Gap: {(training_acc - validation_acc):.2f}%")

            if early_stopping.stop:
                print(f"Stopped early at epoch {epoch+1}")
                break



Starting to train: LR=0.001, WD=0.0001
Epoch 5: Train Loss 3.6313, Train Acc 14.75%, Val Loss 3.4315, Val Acc 19.34%, Dropout Gap: -4.59%
Epoch 10: Train Loss 3.1859, Train Acc 22.16%, Val Loss 3.0985, Val Acc 24.48%, Dropout Gap: -2.32%
Epoch 15: Train Loss 2.8600, Train Acc 28.36%, Val Loss 2.7752, Val Acc 31.42%, Dropout Gap: -3.06%


In [ ]:
import matplotlib.pyplot as plt

# Using the data from your best run (LR=0.001, WD=0.0001)
plt.figure(figsize=(12, 5))

# Plot Accuracy
plt.subplot(1, 2, 1)
plt.plot(best_history['training_acc'], label='Train Accuracy', color='blue')
plt.plot(best_history['validation_acc'], label='Val Accuracy', color='orange')
plt.title('Training and Validation Accuracy \nAccuracy: LR=0.001, WD=0.0001')
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.legend()

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(best_history['training_loss'], label='Train Loss', color='blue')
plt.plot(best_history['validation_loss'], label='Val Loss', color='orange')
plt.title('Training and Validation Loss \nLoss: LR=0.001, WD=0.0001')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()